In [7]:
from datasets import load_dataset, ClassLabel

label_feature = ClassLabel(
    num_classes=3,
    names=["low", "moderate", "high"]
)

dataset = load_dataset(
    "json",
    data_files="train_data.jsonl"
)["train"]

dataset = dataset.cast_column("label", label_feature)

dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42,
    stratify_by_column="label"
)


Generating train split: 0 examples [00:00, ? examples/s]

Casting the dataset:   0%|          | 0/1998 [00:00<?, ? examples/s]

In [9]:
train_ds = dataset["train"]
valid_ds = dataset["test"]

print(train_ds)
print(valid_ds)

Dataset({
    features: ['text', 'label'],
    num_rows: 1598
})
Dataset({
    features: ['text', 'label'],
    num_rows: 400
})


In [10]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "beomi/KcELECTRA-base-v2022"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={0: "low", 1: "moderate", 2: "high"},
    label2id={"low": 0, "moderate": 1, "high": 2}
)


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at beomi/KcELECTRA-base-v2022 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_ds = train_ds.map(tokenize, batched=True)
valid_ds = valid_ds.map(tokenize, batched=True)

train_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)
valid_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)


Map:   0%|          | 0/1598 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [12]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        labels=[0, 1, 2],
        average=None
    )

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "precision_high": precision[2],
        "recall_high": recall[2],
        "f1_high": f1[2],
    }


In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./kc_electra_risk",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
)


In [14]:
# 학습
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()


C:\Users\gjm50\AppData\Local\Temp\ipykernel_2312\1950182647.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision High,Recall High,F1 High
1,0.227000,0.107176,0.980000,0.981250,1.000000,1.000000,1.000000
2,0.068200,0.063113,0.990000,0.990625,1.000000,1.000000,1.000000
3,0.028300,0.063603,0.987500,0.988272,1.000000,1.000000,1.000000


TrainOutput(global_step=300, training_loss=0.20336987614631652, metrics={'train_runtime': 25.3417, 'train_samples_per_second': 189.174, 'train_steps_per_second': 11.838, 'total_flos': 315341431147008.0, 'train_loss': 0.20336987614631652, 'epoch': 3.0})

In [15]:
trainer.evaluate()

{'eval_loss': 0.06311321258544922,
 'eval_accuracy': 0.99,
 'eval_f1_macro': 0.990625,
 'eval_precision_high': 1.0,
 'eval_recall_high': 1.0,
 'eval_f1_high': 1.0,
 'eval_runtime': 0.5224,
 'eval_samples_per_second': 765.762,
 'eval_steps_per_second': 24.887,
 'epoch': 3.0}

In [16]:
from sklearn.metrics import confusion_matrix

pred = trainer.predict(valid_ds)
y_true = pred.label_ids
y_pred = pred.predictions.argmax(axis=1)

print(confusion_matrix(y_true, y_pred))


[[126   3   0]
 [  1 158   0]
 [  0   0 112]]


In [21]:
print(trainer.state.best_model_checkpoint)

./kc_electra_risk\checkpoint-200


In [ ]:
#저장된 체크포인트 모델 불러오기
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

checkpoint_dir = "./kc_electra_risk/checkpoint-200"  # 또는 checkpoint-XXXX

tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

ElectraForSequenceClassification(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(54343, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): L

In [28]:
# 평가
import torch

test_sentences = [
    # 🔹 일상 피로 / 번아웃 (Low가 정상)
    "요즘 너무 지쳐서 그냥 하루가 빨리 끝났으면 좋겠어",
    "아무 생각 없이 좀 쉬고 싶다",
    "하루하루가 너무 반복되는 느낌이야",
    "그냥 피곤해서 말하기도 귀찮아",

    # 🔹 의미 없는 단문 (무조건 Low가 정상)
    "안녕",
    "배고파",
    "졸려",
    "음",
    "아 네",

    # 🔹 실존적 혼란 (Moderate가 정상)
    "사는 의미가 뭔지 잘 모르겠어",
    "이렇게 사는 게 맞나 싶어",
    "앞으로 어떻게 될지 감이 안 와",
    "내가 뭘 원하는지도 모르겠어",

    # 🔹 관계 / 자존감 저하 (Moderate가 정상)
    "나 없어도 다들 잘 사는 것 같아",
    "괜히 민폐만 끼치는 사람인 느낌이야",
    "점점 투명해지는 기분이 들어",
    "아무도 나를 신경 안 쓰는 것 같아",

    # 🔹 은유적 표현 (Moderate or Low가 정상)
    "이 상황에서 도망치고 싶은 마음이 들긴 해",
    "다 내려놓고 그냥 쉬고 싶다",
    "머리가 죽을 것 같아",
    "멘탈이 완전 나간 하루였어",

    # 🔹 borderline (여기서 High 나오면 threshold 낮은 상태)
    "이제는 버티는 게 너무 힘들다",
    "계속 이런 생각이 반복돼서 좀 무섭다",
    "너의 생각은 어때?"
]

inputs = tokenizer(
    test_sentences,
    return_tensors="pt",
    padding=True,
    truncation=True
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)

results = []

for sent, prob in zip(test_sentences, probs):
    high_prob = prob[2].item()

    # 🔥 여기!
    if high_prob > 0.9:
        label_id = 2  # high
    else:
        label_id = torch.argmax(prob[:2]).item()  # low vs moderate

    results.append((sent, label_id, high_prob))

for sent, label_id, high_prob in results:
    print(f"[{model.config.id2label[label_id]} | high_prob={high_prob:.3f}] {sent}")


[low | high_prob=0.039] 요즘 너무 지쳐서 그냥 하루가 빨리 끝났으면 좋겠어
[low | high_prob=0.010] 아무 생각 없이 좀 쉬고 싶다
[low | high_prob=0.016] 하루하루가 너무 반복되는 느낌이야
[low | high_prob=0.584] 그냥 피곤해서 말하기도 귀찮아
[moderate | high_prob=0.311] 안녕
[low | high_prob=0.360] 배고파
[low | high_prob=0.278] 졸려
[moderate | high_prob=0.420] 음
[moderate | high_prob=0.403] 아 네
[moderate | high_prob=0.008] 사는 의미가 뭔지 잘 모르겠어
[moderate | high_prob=0.008] 이렇게 사는 게 맞나 싶어
[moderate | high_prob=0.068] 앞으로 어떻게 될지 감이 안 와
[moderate | high_prob=0.017] 내가 뭘 원하는지도 모르겠어
[moderate | high_prob=0.016] 나 없어도 다들 잘 사는 것 같아
[moderate | high_prob=0.015] 괜히 민폐만 끼치는 사람인 느낌이야
[moderate | high_prob=0.039] 점점 투명해지는 기분이 들어
[moderate | high_prob=0.034] 아무도 나를 신경 안 쓰는 것 같아
[moderate | high_prob=0.023] 이 상황에서 도망치고 싶은 마음이 들긴 해
[low | high_prob=0.015] 다 내려놓고 그냥 쉬고 싶다
[low | high_prob=0.109] 머리가 죽을 것 같아
[low | high_prob=0.886] 멘탈이 완전 나간 하루였어
[moderate | high_prob=0.113] 이제는 버티는 게 너무 힘들다
[moderate | high_prob=0.008] 계속 이런 생각이 반복돼서 좀 무섭다
[moderate | high_prob=0.539] 너의 생각